In [4]:

from dotenv import load_dotenv, find_dotenv
from langsmith import traceable
from pydantic import BaseModel, Field
from typing import List
import os
import re
import inspect

from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_openai import ChatOpenAI

from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.caches import InMemoryCache
from langchain_core.messages import (
    AIMessage, 
    HumanMessage, 
    SystemMessage,
    ToolMessage
)
from langchain_core.prompts import (
    ChatPromptTemplate,
    PromptTemplate,
    SystemMessagePromptTemplate,
    AIMessagePromptTemplate,
    HumanMessagePromptTemplate,
    MessagesPlaceholder
)
import langchain

from langchain_community.chat_message_histories.in_memory import ChatMessageHistory
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import *
from langchain_text_splitters import CharacterTextSplitter

from langchain.tools import tool
from langchain.agents import create_agent



In [5]:
load_dotenv(find_dotenv())
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [27]:
llm = ChatOpenAI(
    model="gpt-5-nano",
    openai_api_key=OPENAI_API_KEY,
    temperature=0,
)

In [ ]:

from langchain_tavily import TavilySearch
from pydantic import BaseModel, Field
from typing import List
from langchain.agents import create_agent
from langchain.tools import tool

In [6]:
MAX_ITERATIONS = 10
MODEL = "qwen3:1.7b"

In [15]:
@traceable(run_type="tool")
def get_product_price(product: str) -> float:
    """Look up the price of a product in the catalog."""
    print(f"    >> Executing get_product_price(product='{product}')")
    prices = {"laptop": 1299.99, "headphones": 149.95, "keyboard": 89.50}
    return prices.get(product, 0)


@traceable(run_type="tool")
def apply_discount(price: float, discount_tier: str) -> float:
    """Apply a discount tier to a price and return the final price. Available tiers: bronze, silver, gold."""
    print(f"    >> Executing apply_discount(price={price}, discount_tier='{discount_tier}')")
    price = float(price)
    discount_percentages = {"bronze": 5, "silver": 12, "gold": 23}
    discount = discount_percentages.get(discount_tier, 0)
    return round(price * (1 - discount / 100), 2)

In [16]:
tools = {
    "get_product_price": get_product_price,
    "apply_discount": apply_discount,
}


In [23]:

def get_tool_descriptions(tools_dict):
    descriptions = []
    for tool_name, tool_function in tools_dict.items():
        # __wrapped__ bypasses decorator wrappers (e.g., @traceable adds *, config=None)
        original_function = getattr(tool_function, "__wrapped__", tool_function)
        signature = inspect.signature(original_function)
        docstring = inspect.getdoc(tool_function) or ""
        descriptions.append(f"{tool_name}{signature} - {docstring}")
    return "\n".join(descriptions)

In [24]:
tool_descriptions = get_tool_descriptions(tools)
tool_names = ", ".join(tools.keys())

In [25]:
print(tool_descriptions)
print('_' *90)
print(tool_names)

get_product_price(product: str) -> float - Look up the price of a product in the catalog.
apply_discount(price: float, discount_tier: str) -> float - Apply a discount tier to a price and return the final price. Available tiers: bronze, silver, gold.
__________________________________________________________________________________________
get_product_price, apply_discount


In [26]:

react_prompt = f"""
STRICT RULES — you must follow these exactly:
1. NEVER guess or assume any product price. You MUST call get_product_price first to get the real price.
2. Only call apply_discount AFTER you have received a price from get_product_price. Pass the exact price returned by get_product_price — do NOT pass a made-up number.
3. NEVER calculate discounts yourself using math. Always use the apply_discount tool.
4. If the user does not specify a discount tier, ask them which tier to use — do NOT assume one.

Answer the following questions as best you can. You have access to the following tools:

{tool_descriptions}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action, as comma separated values
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {{question}}
Thought:"""

In [ ]:
@traceable(name="Ollama Agent Loop")
def run_agent(question: str):
    print(f"Question: {question}")
    print("=" * 60)

    # CHANGE 5: One prompt string replaces the system/user message split.
    prompt = react_prompt.format(question=question)
    scratchpad = "" 
    
    for iteration in range(1, MAX_ITERATIONS + 1):
        print(f"\n--- Iteration {iteration} ---")
        full_prompt = prompt + scratchpad

        response = llm.invoke(full_prompt)

        output = response.content
        print(f"LLM Output:\n{output}")

        # ------------------------
        # FINAL ANSWER
        # ------------------------
        final_answer_match = re.search(
            r"Final Answer:\s*(.+)",
            output,
            re.DOTALL,
        )

        if final_answer_match:
            final_answer = final_answer_match.group(1).strip()

            print("\n" + "=" * 60)
            print(f"Final Answer: {final_answer}")

            return final_answer

        # ------------------------
        # ACTION PARSE
        # ------------------------

        action_match = re.search(
            r"Action:\s*(.+)",
            output
        )

        action_input_match = re.search(
            r"Action Input:\s*(.+)",
            output
        )

        if not action_match or not action_input_match:

            print("ERROR: Could not parse action")
            break

        tool_name = action_match.group(1).strip()
        tool_input_raw = action_input_match.group(1).strip()

        print(f"Tool Selected: {tool_name}")
        print(f"Args raw: {tool_input_raw}")

        raw_args = [
            x.strip()
            for x in tool_input_raw.split(",")
        ]

        args = [
            x.split("=", 1)[-1]
            .strip()
            .strip("'\"")
            for x in raw_args
        ]

        # ------------------------
        # TOOL EXECUTION
        # ------------------------

        if tool_name not in tools:

            observation = (
                f"Error: Tool '{tool_name}' not found. "
                f"Available tools: {list(tools.keys())}"
            )

        else:

            observation = str(
                tools[tool_name](*args)
            )

        print(f"Observation: {observation}")

        # ------------------------
        # SCRATCHPAD UPDATE
        # ------------------------

        scratchpad += (
            f"{output}\n"
            f"Observation: {observation}\n"
            f"Thought:"
        )

    print("ERROR: Max iterations reached")

    return None
        

In [ ]:
def run_agent(question: str):
    tools = [get_product_price, apply_discount]
    tools_dict = {t.name: t for t in tools}

    # Initialize OpenAI LLM
    llm = ChatOpenAI(model="gpt-5-nano", openai_api_key=OPENAI_API_KEY, temperature=0)
    llm_with_tools = llm.bind_tools(tools)

    print(f"Question: {question}")
    print("=" * 60)

    messages = [
        SystemMessage(
            content=(
                "You are a helpful shopping assistant. "
                "You have access to a product catalog tool "
                "and a discount tool.\n\n"
                "STRICT RULES — you must follow these exactly:\n"
                "1. NEVER guess or assume any product price. "
                "You MUST call get_product_price first to get the real price.\n"
                "2. Only call apply_discount AFTER you have received "
                "a price from get_product_price. Pass the exact price "
                "returned by get_product_price — do NOT pass a made-up number.\n"
                "3. NEVER calculate discounts yourself using math. "
                "Always use the apply_discount tool.\n"
                "4. If the user does not specify a discount tier, "
                "ask them which tier to use — do NOT assume one."
            )
        ),
        HumanMessage(content=question),
    ]

    for iteration in range(1, MAX_ITERATIONS + 1):
        print(f"\n--- Iteration {iteration} ---")

        ai_message = llm_with_tools.invoke(messages)

        tool_calls = ai_message.tool_calls

        # If no tool calls, this is the final answer
        if not tool_calls:
            print(f"\nFinal Answer: {ai_message.content}")
            return ai_message.content

        # Process only the FIRST tool call — force one tool per iteration
        tool_call = tool_calls[0]
        tool_name = tool_call.get("name")
        tool_args = tool_call.get("args", {})
        tool_call_id = tool_call.get("id")

        print(f"  [Tool Selected] {tool_name} with args: {tool_args}")

        tool_to_use = tools_dict.get(tool_name)
        if tool_to_use is None:
            raise ValueError(f"Tool '{tool_name}' not found")

        observation = tool_to_use.invoke(tool_args)

        print(f"  [Tool Result] {observation}")

        messages.append(ai_message)
        messages.append(
            ToolMessage(content=str(observation), tool_call_id=tool_call_id)
        )

    print("ERROR: Max iterations reached without a final answer")
    return None

In [ ]:
result = run_agent("What is the price of a laptop after applying a gold discount?")